# Day 12 — Perspektif Düzeltme ve Homografi
## Kamera Açısı Bozulmaları, Köşe Sıralama ve Projektif Dönüşüm ile Kuşbakışı Hizalama

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** Perspektif Düzeltme ve Homografi (Yaprak 23 & 24)

### 1. Problem
Kalite kontrol hattında tezgâh üzerindeki alan darlığı nedeniyle kameralar halı yüzeyine her zaman tam dik (ortogonal) bakamaz. Eğik açıyla yerleştirilen kameralardan alınan görüntülerde perspektif yamulması (trapezoid bozulma) meydana gelir ve desenlerin simetri/ölçü analizi yapılamaz.

### 2. Why the Problem Matters
Projektif geometri ve 3x3 homografi matrisi kullanılarak 4 köşe noktası düzlem üzerinde istenen en/boy oranındaki dikdörtgene (kuşbakışı / top-down) izdüşürülür. Bu sayede fiziksel ölçümler milimetrik hassasiyetle yapılabilir.

### 3. Engineering Concepts
- **Köşe Sıralama Algoritması**: Dört köşe noktasının saat yönünde [Sol-Üst, Sağ-Üst, Sağ-Alt, Sol-Alt] olarak sıralanması.
- **Homografi Matrisi ($H_{3\times3}$)**: Düzlemler arası projektif dönüşüm matrisi.
- **Warp Perspective**: Piksel interpolasyonu ile görüntünün düzeltilmiş hedef koordinat sistemine ötelenmesi.

In [ ]:
# 4. Library / API Investigation
import cv2
import numpy as np
from day12.mini_project.src.homography_rectifier import HomographyRectifier, order_four_points

rectifier = HomographyRectifier(output_width=300, output_height=400)
print("Homografi Düzeltici Hazır.")

In [ ]:
# 5. Minimal Implementation
# Perspektif bozulmalı sentetik dörtgen ve halı deseni
img = np.zeros((500, 500, 3), dtype=np.uint8)
cv2.circle(img, (250, 250), 100, (0, 165, 255), -1) # Merkez madalyon

# Eğik kamera açısı dörtgeni
distorted_corners = np.array([
    [60, 80],    # Sol-Üst
    [420, 120],  # Sağ-Üst
    [390, 440],  # Sağ-Alt
    [90, 420]    # Sol-Alt
], dtype=np.float32)

warped, H = rectifier.rectify(img, distorted_corners)
print("Homografi Matrisi (H):\n", np.round(H, 4))

In [ ]:
# 6. Experiment: Boyut Doğrulaması
print(f"Orijinal Boyut: {img.shape} -> Düzeltilmiş Kuşbakışı Boyut: {warped.shape}")

In [ ]:
# 7. Visualization: Perspektif Öncesi ve Sonrası
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].plot([60, 420, 390, 90, 60], [80, 120, 440, 420, 80], color="red", linewidth=2)
axes[0].set_title("Eğik Açı Kamera Girdisi")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title("Düzeltilmiş Kuşbakışı (Top-Down)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert warped.shape == (400, 300, 3)
assert H.shape == (3, 3)
print("Projektif homografi doğrulaması başarılı.")

In [ ]:
# 9. Failure Cases: Çakışık veya tanımsız köşe noktaları
degenerate_corners = np.array([[0, 0], [0, 0], [10, 10], [10, 10]], dtype=np.float32)
try:
    rectifier.rectify(img, degenerate_corners)
except Exception as e:
    print("Beklenen dejenerasyon hatası yakalandı:", type(e).__name__)

### 10. Conclusions
Perspektif düzeltme ve homografi yöntemleri uygulanmış, eğik kamera açılarından alınan halı görüntüleri ortogonal kuşbakışı koordinat sistemine taşınmıştır.